# FrameCheck — Train a Real-World AI Image Detector

Fine-tune `google/vit-base-patch16-224` on real photographs and AI-generated images.
This notebook uses the public Hugging Face dataset `TheKernel01/Tiny-GenImage`, so
no Kaggle account or API key is needed.

**Dataset:** 35,000 images, with real/fake labels and examples from several generators.
The dataset card lists a CC BY-NC-SA 4.0 license (non-commercial use only); check the
license and upstream provenance before publishing a model or using it commercially.

**Runtime:** Use a Colab T4 GPU. Downloading the dataset is about 8.3 GB; training time
depends on Colab availability and the selected sample cap.

The official `train` split is used for training and validation. The dataset's official
`validation` split is kept untouched for final test metrics.

## Export
The trained model is saved to Google Drive and as a zip file for local use. Hugging Face
Hub upload is optional and requires your own token; it is not required to train or use
the model locally.


## 0. GPU check + install

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError('No GPU found! Go to Runtime → Change runtime type → T4 GPU.')
print('GPU:', result.stdout.strip())

!pip install -q transformers datasets accelerate evaluate scikit-learn timm pillow tqdm

## 1. Configuration — edit these values

In [ ]:
# ── Model ────────────────────────────────────────────────────────────────────
BASE_MODEL   = 'google/vit-base-patch16-224'
OUTPUT_NAME  = 'framecheck-ai-detector'

# ── Hugging Face dataset ─────────────────────────────────────────────────────
DATASET_ID   = 'TheKernel01/Tiny-GenImage'

# ── Optional Hub upload (leave blank to skip) ─────────────────────────────────
HF_USERNAME  = ''
# If you want to publish the model, authenticate with your own HF token in Colab.

# ── Training ─────────────────────────────────────────────────────────────────
IMAGE_SIZE   = 224
BATCH_SIZE   = 32     # reduce to 16 if Colab runs out of GPU memory
EPOCHS       = 4
LR           = 2e-5
SEED         = 42
MAX_SAMPLES  = 14_000 # maximum images per class from the training split

# ── Google Drive output ──────────────────────────────────────────────────────
SAVE_TO_DRIVE = True
DRIVE_PATH    = '/content/drive/MyDrive/framecheck_model'

print('Config loaded.')
print(f'Dataset    : {DATASET_ID}')
print(f'Epochs     : {EPOCHS}   LR: {LR}   Batch: {BATCH_SIZE}')


## 2. Load the Hugging Face dataset

The dataset card defines `label=0` as real and `label=1` as fake. The code below reads
the class names from the dataset metadata and checks them instead of assuming a label
ordering. The dataset has official `train` and `validation` splits; the latter is reserved
for final testing and is never used for training or checkpoint selection.


In [ ]:
from datasets import load_dataset

print(f'Loading {DATASET_ID} from Hugging Face...')
hf_data = load_dataset(DATASET_ID)
if 'train' not in hf_data or 'validation' not in hf_data:
    raise RuntimeError(f'Expected train and validation splits; received: {list(hf_data.keys())}')

train_source = hf_data['train']
test_source = hf_data['validation']
label_feature = train_source.features['label']
label_names = [str(name).strip().lower() for name in label_feature.names]
try:
    REAL_ID = label_names.index('real')
    FAKE_ID = label_names.index('fake')
except ValueError as exc:
    raise RuntimeError(f'Expected label names real/fake, found: {label_names}') from exc

print('Label names:', label_names)
print(f'Rows: train={len(train_source):,}, official test={len(test_source):,}')
for split_name, split in [('train', train_source), ('official test', test_source)]:
    counts = {name.upper(): sum(int(y) == idx for y in split['label'])
              for idx, name in enumerate(label_names)}
    print(f'{split_name} class counts:', counts)


## 3. Create reproducible train / validation / test loaders

We make a stratified validation subset from the official training split. The Hugging Face
validation split remains untouched as the final test set. Labels are converted to the
backend contract: `0=FAKE`, `1=REAL`.


In [ ]:
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from transformers import ViTImageProcessor
from sklearn.model_selection import train_test_split

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

train_labels_raw = [int(value) for value in train_source['label']]
real_candidates = [i for i, value in enumerate(train_labels_raw) if value == REAL_ID]
fake_candidates = [i for i, value in enumerate(train_labels_raw) if value == FAKE_ID]
n_per_class = min(len(real_candidates), len(fake_candidates), MAX_SAMPLES or len(train_source))
if n_per_class < 1:
    raise RuntimeError('No usable real/fake training examples were found.')

selected = random.sample(real_candidates, n_per_class) + random.sample(fake_candidates, n_per_class)
selected_labels = [train_labels_raw[i] for i in selected]
train_idx, val_idx = train_test_split(
    selected, test_size=0.10, stratify=selected_labels, random_state=SEED
)
test_idx = list(range(len(test_source)))
print(f'Train: {len(train_idx):,}  Validation: {len(val_idx):,}  Held-out test: {len(test_idx):,}')
print('Label mapping: 0=FAKE, 1=REAL')

processor = ViTImageProcessor.from_pretrained(BASE_MODEL)

class ImageDataset(Dataset):
    def __init__(self, source, indices, augment=False):
        self.source = source
        self.indices = list(indices)
        self.aug = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        ]) if augment else None

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        row = self.source[self.indices[index]]
        image = row['image'].convert('RGB')
        if self.aug:
            image = self.aug(image)
        pixel_values = processor(images=image, return_tensors='pt')['pixel_values'].squeeze(0)
        raw_label = int(row['label'])
        label = 1 if raw_label == REAL_ID else 0
        return pixel_values, torch.tensor(label, dtype=torch.long)

trn_ds = ImageDataset(train_source, train_idx, augment=True)
val_ds = ImageDataset(train_source, val_idx, augment=False)
tst_ds = ImageDataset(test_source, test_idx, augment=False)

trn_dl = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
tst_dl = DataLoader(tst_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('Dataloaders ready.')


## 4. Load model and train

In [ ]:
from transformers import ViTForImageClassification, get_cosine_schedule_with_warmup
from torch.optim import AdamW
import torch.nn as nn

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

id2label = {0: 'FAKE', 1: 'REAL'}
label2id = {'FAKE': 0, 'REAL': 1}

model = ViTForImageClassification.from_pretrained(
    BASE_MODEL,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
).to(DEVICE)

print(f'Parameters: {model.num_parameters():,}')

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(trn_dl) * EPOCHS
warmup_steps = total_steps // 10
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
criterion = nn.CrossEntropyLoss()

def evaluate(dl):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    all_preds, all_labels_list = [], []
    with torch.no_grad():
        for px, lbl in dl:
            px, lbl = px.to(DEVICE), lbl.to(DEVICE)
            out   = model(pixel_values=px)
            loss  = criterion(out.logits, lbl)
            preds = out.logits.argmax(dim=-1)
            loss_sum += loss.item() * len(lbl)
            correct  += (preds == lbl).sum().item()
            total    += len(lbl)
            all_preds.extend(preds.cpu().tolist())
            all_labels_list.extend(lbl.cpu().tolist())
    return loss_sum/total, correct/total, all_preds, all_labels_list

best_val_acc = 0.0
best_ckpt = '/content/best_checkpoint.pt'

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss, running_correct, running_total = 0.0, 0, 0

    for step, (px, lbl) in enumerate(trn_dl, 1):
        px, lbl = px.to(DEVICE), lbl.to(DEVICE)
        optimizer.zero_grad()
        out  = model(pixel_values=px)
        loss = criterion(out.logits, lbl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running_loss    += loss.item() * len(lbl)
        running_correct += (out.logits.argmax(-1) == lbl).sum().item()
        running_total   += len(lbl)

        if step % 50 == 0:
            print(f'  Epoch {epoch} step {step}/{len(trn_dl)}  '
                  f'loss={running_loss/running_total:.4f}  '
                  f'acc={running_correct/running_total:.4f}')

    val_loss, val_acc, _, _ = evaluate(val_dl)
    print(f'\nEpoch {epoch}/{EPOCHS}  val_loss={val_loss:.4f}  val_acc={val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_ckpt)
        print(f'  → New best: {best_val_acc:.4f} — checkpoint saved')

print(f'\nTraining done. Best val accuracy: {best_val_acc:.4f}')

## 5. Evaluate on held-out test set (no data leakage)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# Load best checkpoint
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))

# Collect predictions and probabilities
model.eval()
all_preds, all_labels_list, all_probs = [], [], []
with torch.no_grad():
    for px, lbl in tst_dl:
        px = px.to(DEVICE)
        out = model(pixel_values=px)
        probs = F.softmax(out.logits, dim=-1)
        all_preds.extend(out.logits.argmax(-1).cpu().tolist())
        all_labels_list.extend(lbl.tolist())
        all_probs.extend(probs[:, 1].cpu().tolist())  # P(REAL)

print('=== TEST SET RESULTS (never seen during training) ===')
print(classification_report(all_labels_list, all_preds,
                             target_names=['FAKE (0)', 'REAL (1)'], digits=4))
roc = roc_auc_score(all_labels_list, all_probs)
print(f'ROC-AUC: {roc:.4f}')

cm = confusion_matrix(all_labels_list, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['FAKE', 'REAL'], yticklabels=['FAKE', 'REAL'], ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion matrix — test set')
plt.tight_layout(); plt.show()

## 6. Save model + export for FrameCheck

In [ ]:
import shutil
from pathlib import Path


SAVE_DIR = Path(f'/content/{OUTPUT_NAME}')
SAVE_DIR.mkdir(exist_ok=True)

# Restore best checkpoint and save the Hugging Face model + matching image processor.
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
model.save_pretrained(str(SAVE_DIR))
processor.save_pretrained(str(SAVE_DIR))
print(f'Model saved to {SAVE_DIR}')

# Create a portable zip for downloading into the FrameCheck project.
MODEL_ZIP = shutil.make_archive(f'/content/{OUTPUT_NAME}', 'zip', root_dir=SAVE_DIR)
print(f'Portable model archive: {MODEL_ZIP}')

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dst = Path(DRIVE_PATH)
    drive_dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(SAVE_DIR), str(drive_dst), dirs_exist_ok=True)
    shutil.copy2(MODEL_ZIP, drive_dst.parent / f'{OUTPUT_NAME}.zip')
    print(f'Model folder saved to Google Drive: {DRIVE_PATH}')
    print(f'Zip saved to Google Drive: {drive_dst.parent / (OUTPUT_NAME + ".zip")}')

from google.colab import files
files.download(MODEL_ZIP)


## 7. (Optional) Push to HuggingFace Hub

In [ ]:
if HF_USERNAME:
    from huggingface_hub import notebook_login
    notebook_login()  # enter your HF token when prompted

    repo_id = f'{HF_USERNAME}/{OUTPUT_NAME}'
    model.push_to_hub(repo_id)
    processor.push_to_hub(repo_id)
    print(f'Pushed to: https://huggingface.co/{repo_id}')
    print()
    print('To use this model in FrameCheck, update backend/app.py:')
    print(f'  HF_MODEL_ID = "{repo_id}"')
    print('OR set the environment variable:')
    print(f'  FRAMECHECK_HF_MODEL={repo_id}')
else:
    print('HF_USERNAME is blank — skipping Hub upload.')
    print('Set HF_USERNAME in Section 1 and re-run this cell to push.')

## 8. Use the trained model in FrameCheck

### Local model folder
Download `framecheck-ai-detector.zip` from Colab and extract it to:
`models/framecheck-ai-detector/`

Then start the backend from the project root in PowerShell:
```powershell
$env:FRAMECHECK_HF_MODEL='models/framecheck-ai-detector'
.venv\Scripts\python -m uvicorn backend.app:app --host 127.0.0.1 --port 8000
```

The API already loads local Hugging Face image-classification folders through
`FRAMECHECK_HF_MODEL`. The trained class mapping is FAKE=0, REAL=1.

The dataset license is CC BY-NC-SA 4.0. Do not use this dataset or resulting model
commercially without confirming you have the required rights.
